In [1]:
import sys
from pathlib import Path
here = Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "src" / "model.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.data import ORDERS_PATH, TARGET
from src.model import FEATURES, time_split, make_logistic, score

orders = pd.read_csv(ORDERS_PATH, parse_dates=["order_date"])
train, test = time_split(orders)
for name, part in [("train", train), ("test", test)]:
    print(f"{name}: {len(part):>6} orders | {part['order_date'].min():%Y-%m-%d} → {part['order_date'].max():%Y-%m-%d} | late {part[TARGET].mean():.3f}")

train:  50317 orders | 2015-01-01 → 2017-07-10 | late 0.573
test:  12580 orders | 2017-07-10 → 2018-01-31 | late 0.574


In [2]:
results = []

# 1. Always say "late" (the most common answer)
results.append(score("always late", test[TARGET], np.ones(len(test))))

# 2. Look up the late rate of the order's shipping mode, learned from TRAIN only
mode_rate = train.groupby("shipping_mode")[TARGET].mean()
print(mode_rate.round(3))
results.append(score("shipping-mode lookup", test[TARGET], test["shipping_mode"].map(mode_rate)))

pd.DataFrame(results)

shipping_mode
First Class       1.000
Same Day          0.471
Second Class      0.799
Standard Class    0.399
Name: late, dtype: float64


,model,accuracy,roc_auc
0,always late,0.5745,0.5000
1,shipping-mode lookup,0.6953,0.7443


In [3]:
logit = make_logistic()
logit.fit(train[FEATURES], train[TARGET])
prob = logit.predict_proba(test[FEATURES])[:, 1]   # column 1 = probability of "late"
results.append(score("logistic regression", test[TARGET], prob))
pd.DataFrame(results)

,model,accuracy,roc_auc
0,always late,0.5745,0.5000
1,shipping-mode lookup,0.6953,0.7443
2,logistic regression,0.7062,0.7528
